In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, FloatType, LongType

spark = SparkSession.builder\
        .appName("e-commerce_analysis")\
        .master("local[*]")\
        .config("spark.executor.memory", "2g")\
        .config("spark.driver.memory", "1g")\
        .getOrCreate()

In [ ]:
# data_path = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-subset-10p-enriched.csv"
data_path = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-enriched.csv"
output_dir = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/parquet/"

In [15]:
schema = StructType([
    StructField("timestamp", DateType()),
    StructField("event_type", StringType()),
    StructField("product_id", IntegerType()),
    StructField("category_id", LongType()),
    StructField("category_code", StringType()),
    StructField("brand", StringType()),
    StructField("price", FloatType()),
    StructField("user_id", LongType()),
    StructField("user_session", StringType()),
    StructField("name", StringType()),
    StructField("username", StringType()),
    StructField("email", StringType()),
    StructField("address", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("state_code", StringType()),
    StructField("zip_code", StringType()),
    StructField("sex", StringType()),
    StructField("product_name", StringType()),
    StructField("product_description", StringType())
])

In [16]:
df = spark.read.format("csv")\
    .schema(schema)\
    .option("header", "true")\
    .load(data_path)\
    .select("timestamp", "event_type", "product_id", "price", "user_id", "user_session")
print("Number of partitions:", df.rdd.getNumPartitions())
df.cache()

Number of partitions: 12


DataFrame[timestamp: date, event_type: string, product_id: int, price: float, user_id: bigint, user_session: string]

In [ ]:
df.orderBy(["user_session", "product_id"]).take(30)

25/10/27 06:46:23 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: event_time, event_type, product_id, price, user_id, user_session
 Schema: timestamp, event_type, product_id, price, user_id, user_session
Expected: timestamp but found: event_time
CSV file: file:///home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-subset-10p-enriched.csv


[Row(timestamp=datetime.date(2019, 10, 3), event_type='view', product_id=2501061, price=195.4199981689453, user_id=512483064, user_session='000003eb-b63e-45d9-9f26-f229057c654a'),
 Row(timestamp=datetime.date(2019, 10, 3), event_type='view', product_id=1004739, price=196.97999572753906, user_id=556327911, user_session='000011ef-e9fc-4920-96cd-2534bda3cdc2'),
 Row(timestamp=datetime.date(2019, 10, 3), event_type='view', product_id=1004739, price=196.97999572753906, user_id=556327911, user_session='000011ef-e9fc-4920-96cd-2534bda3cdc2'),
 Row(timestamp=datetime.date(2019, 10, 3), event_type='view', product_id=1004739, price=196.94000244140625, user_id=556327911, user_session='000011ef-e9fc-4920-96cd-2534bda3cdc2'),
 Row(timestamp=datetime.date(2019, 10, 3), event_type='view', product_id=1004741, price=185.6699981689453, user_id=556327911, user_session='000011ef-e9fc-4920-96cd-2534bda3cdc2'),
 Row(timestamp=datetime.date(2019, 10, 3), event_type='view', product_id=1004741, price=185.66999

In [18]:
df.createOrReplaceTempView("event_log")

In [19]:
# TODO Create one table containing all view records
view_df = spark.sql("SELECT DISTINCT * FROM event_log WHERE event_type = 'view'")

In [20]:
# TODO Create one table containing all cart & purchase records
engagement_df = spark.sql("SELECT DISTINCT * FROM event_log WHERE event_type != 'view'")

In [21]:
# TODO Join the two tables on user_session and product_id
view_df.join(engagement_df, on=["user_session", "product_id"], how="inner").orderBy("user_session").show(20)

+--------------------+----------+----------+----------+------+---------+----------+----------+------+---------+
|        user_session|product_id| timestamp|event_type| price|  user_id| timestamp|event_type| price|  user_id|
+--------------------+----------+----------+----------+------+---------+----------+----------+------+---------+
|0001c7bd-3921-471...|   1801881|2019-10-03|      view|506.19|555235633|2019-10-03|      cart|506.19|555235633|
|00020fdd-4644-4a3...|  29502152|2019-10-03|      view|  9.01|513287373|2019-10-03|  purchase|  9.01|513287373|
|0002c5ea-3509-4d0...|   9200557|2019-10-01|      view|  9.76|541539898|2019-10-01|  purchase|  9.76|541539898|
|0003c729-3377-469...|   4804056|2019-10-03|      view| 161.9|517122869|2019-10-03|  purchase| 161.9|517122869|
|0003f986-6668-447...|   1004858|2019-10-03|      view|132.02|523651282|2019-10-03|      cart|132.02|523651282|
|0003f986-6668-447...|   1004858|2019-10-03|      view|132.02|523651282|2019-10-03|  purchase|132.02|523

In [23]:
# Coalesce into fewer partitions
df.coalesce(5).write.partitionBy("timestamp").parquet(path=output_dir, mode="overwrite")

In [ ]:
spark.stop()